# Stage 2 — Preprocessing (AMZN)

Runs diagnostics and split checks over the AMZN artifacts from Stage 1:

- Tokenization statistics for each NLP model.
- Shared chronological cutoff date across text and numerical tables.
- Chronological train/test split sanity checks for both text and numerical data.

No random train/test split is used in this stage.


In [1]:
# Load shared config, paths, and helper utilities used in all stages.
from common import *

# Tokenizer class used to inspect token lengths for each model.
from transformers import AutoTokenizer

[common] device=cuda  artifacts=/cluster/tufts/c26sp1cee0132/jmonta04/ai_project/module_version/artifacts  results=/cluster/tufts/c26sp1cee0132/jmonta04/ai_project/module_version/results


## 2.1 Load artifacts from Stage 1

In [ ]:
# Load Stage 1 artifacts.
text_df = load_text_df()
num_df = load_num_df()

# Convert to date-only values so all downstream joins/splits align cleanly.
text_df["date"] = pd.to_datetime(text_df["date"]).dt.normalize()
num_df["date"] = pd.to_datetime(num_df["date"]).dt.normalize()

# Choose one shared cutoff date so text and numerical test sets cover
# the same out-of-sample period.
cutoff_date = get_shared_chronological_cutoff(
    text_df=text_df,
    num_df=num_df,
    test_size=CONFIG["regression"]["test_size"],
)

# Quick sanity checks on loaded data size and split point.
print(f"text_df rows: {len(text_df)}")
print(f"num_df rows:  {len(num_df)}")
print(f"Shared chronological cutoff: {cutoff_date.date()}")

## 2.2 Tokenization statistics per NLP model

In [ ]:
# Iterate through each model to understand how long AMZN headlines are
# under that tokenizer.
for model, path in NLP_MODELS.items():
    # Build tokenizer for the current model checkpoint.
    tok = AutoTokenizer.from_pretrained(path)

    # Count raw token length per headline without truncation.
    lengths = text_df["text"].apply(
        lambda text: len(tok(text, truncation=False, padding=False)["input_ids"])
    )

    # max_len is the training-time token cap from config.
    max_len = CONFIG["nlp"]["max_length"]
    # n_trunc counts how many rows would be truncated at training time.
    n_trunc = int((lengths > max_len).sum())

    print(f"Model: {model}")
    print(f"  Num truncated at {max_len}: {n_trunc}")
    print(f"  Min/median/max tokens: {lengths.min()} / {lengths.median():.1f} / {lengths.max()}")
    print("  Percentiles (50/90/95/99/100):")
    print(lengths.quantile([0.5, 0.9, 0.95, 0.99, 1.0]).to_string())
    print()

## 2.3 Chronological split sanity checks

Validates that both text and numerical data use the same train/test cutoff date.


In [ ]:
# Chronological split for text data (train = early dates, test = later dates).
text_train_df, text_test_df = chronological_split_by_date(
    df=text_df,
    date_col="date",
    cutoff_date=cutoff_date,
)

# Chronological split + scaling for numerical data.
(
    X_train,
    X_test,
    y_train,
    y_test,
    _,
    feature_cols,
    num_train_df,
    num_test_df,
) = make_num_splits_chronological(
    df=num_df,
    cutoff_date=cutoff_date,
    target_col="risk_score",
    date_col="date",
)

# Show split boundaries to confirm no future leakage.
print("Text split:")
print(f"  train rows: {len(text_train_df)}")
print(f"  test rows:  {len(text_test_df)}")
print(f"  train max date: {text_train_df['date'].max().date()}")
print(f"  test min date:  {text_test_df['date'].min().date()}")

# Show numerical matrix shapes and date boundaries.
print("\nNumerical split:")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_test shape:  {X_test.shape}")
print(f"  feature count: {len(feature_cols)}")
print(f"  train max date: {num_train_df['date'].max().date()}")
print(f"  test min date:  {num_test_df['date'].min().date()}")